In [1]:
import sys, random
sys.path.append("..")

import numpy as np
from onnxruntime.quantization import quantize_static, QuantFormat, QuantType, CalibrationDataReader
from onnxruntime.quantization.shape_inference import quant_pre_process
from src.dataset import ForestDataset

SRC = "../models/forest_unet.onnx"
PRE = "../models/forest_unet_pre.onnx"
OUT = "../models/forest_unet_int8.onnx"

quant_pre_process(SRC, PRE)

class Reader(CalibrationDataReader):
    def __init__(self, n=40):
        random.seed(0)
        items = []
        for biome in ["amazon", "atlantic"]:
            ds = ForestDataset(biome, "Training")
            for i in random.sample(range(len(ds)), n // 2):
                items.append({"input": ds[i][0][None].numpy()})
        self.it = iter(items)
    def get_next(self):
        return next(self.it, None)

quantize_static(
    PRE, OUT, Reader(),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
)

import os
print("int8 size:", round(os.path.getsize(OUT) / 1e6, 1), "MB")

int8 size: 24.7 MB


In [2]:
import onnxruntime as ort

def make(path):
    return ort.InferenceSession(path, providers=["CPUExecutionProvider"])

def iou(sess, biome, n=30):
    ds = ForestDataset(biome, "Validation")
    tp = fp = fn = 0
    for i in range(min(n, len(ds))):
        x, y = ds[i]
        pred = sess.run(None, {"input": x[None].numpy()})[0][0, 0] > 0
        true = y[0].numpy() > 0.5
        tp += (pred & true).sum(); fp += (pred & ~true).sum(); fn += (~pred & true).sum()
    return tp / (tp + fp + fn)

for name, path in [("fp32", SRC), ("int8", OUT)]:
    s = make(path)
    print(name, {b: round(float(iou(s, b)), 3) for b in ["amazon", "atlantic"]})

fp32 {'amazon': 0.948, 'atlantic': 0.905}
int8 {'amazon': 0.947, 'atlantic': 0.9}
